# Liquid Ensembles: paper analysis

In-depth analysis of the `exp_aggregation` and `exp_ambiguity_gradient` sweeps,
plus the figures used in the paper.

This notebook is a companion to `experiment_aggregation.ipynb`, which was the
exploratory pass. What is different here:

* **Discovery instead of hardcoded folder names.** Run folders are found and
  parsed with a regex, so `exp_ambiguity_gradient` (an experiment name that
  itself contains underscores) is handled correctly.
* **One streaming pass over the metric files.** Instead of materialising a row
  per (run, epoch, seed), the loader keeps three small frames: seed-averaged
  learning curves, seed-level values at the two selection epochs, and one row of
  metadata per run.
* **Matched-block statistics.** Both sweeps use `Pool.paired`, so consecutive
  `run_id`s share a random configuration *and* a PRNG key and differ only in the
  paired factor. Every block is validated rather than assumed, and the paired
  contrasts get bootstrap confidence intervals, a Wilcoxon signed-rank test,
  matched-pairs rank-biserial effect sizes and Holm-corrected p-values.
* **Divergence is treated as an outcome, not as missing data.** Dropping the
  NaN runs silently biases the comparison in favour of whichever variant
  diverges more, so divergence gets its own figure and a worst-case sensitivity
  analysis.
* **Paper-ready figures** written to `paper_plots/` as vector PDF plus PNG.

## A caveat on `validation_performance_loss`

`performance_loss` in `math_utils.py` is

$$\mathbb{E}\left[\sum_i p_i(x)\,\ell_i(x) - \sum_i p_i(x)\,a_i(x)\right]$$

that is, the delegation-weighted loss **minus** the delegation-weighted
ambiguity. The subtraction happens for every value of `ambiguity_gradient`;
only the gradient flow differs (`none` wraps the ambiguity term in
`stop_gradient`, `both` also lets it reach the predictors). So the quantity is
defined identically across the three variants and *is* comparable between them,
but it is a diversity-discounted loss and not a measure of predictive quality:
a model can lower it by disagreeing more. The headline numbers below therefore
use the task metric (accuracy / $R^2$), with the discounted loss reported
alongside. The raw weighted performance term is not currently logged
separately; logging it would make this cleaner.

In [ ]:
import json
import os
import re
import warnings
from dataclasses import dataclass, field
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy import stats

RNG = np.random.default_rng(20260828)

NOTEBOOK_DIR = Path.cwd()
RUNS_DIR = Path(os.environ.get("LIQUID_RUNS_DIR", NOTEBOOK_DIR / "runs")).resolve()
FIG_DIR = Path(os.environ.get("LIQUID_FIG_DIR", NOTEBOOK_DIR / "paper_plots")).resolve()
TABLE_DIR = FIG_DIR / "tables"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("runs   :", RUNS_DIR, "(exists)" if RUNS_DIR.is_dir() else "(MISSING)")
print("figures:", FIG_DIR)

In [ ]:
# --- Paper figure style -----------------------------------------------------
# fonttype 42 embeds real TrueType text in the PDF, so the figures stay
# selectable and searchable in the compiled paper instead of being outlined.

PAPER_STYLE = {
    "figure.dpi": 110,
    "savefig.dpi": 400,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.02,
    "savefig.transparent": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "serif",
    "font.serif": ["DejaVu Serif"],
    "mathtext.fontset": "dejavuserif",
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 9,
    "axes.titleweight": "bold",
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "0.85",
    "grid.linewidth": 0.6,
    "legend.frameon": False,
    "lines.linewidth": 1.4,
    "lines.markersize": 4,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "figure.constrained_layout.use": True,
}

mpl.rcParams.update(PAPER_STYLE)

# Okabe-Ito, colourblind safe.
OKABE_ITO = {
    "blue": "#0072B2",
    "orange": "#E69F00",
    "green": "#009E73",
    "vermillion": "#D55E00",
    "purple": "#CC79A7",
    "sky": "#56B4E9",
    "yellow": "#F0E442",
    "black": "#000000",
}

# Figure widths in inches, for a single-column TMLR page.
#
# FULL_W must equal the LaTeX \textwidth of the tmlr.sty template, so a figure
# included at [width=\textwidth] is reproduced at 1:1 and its fonts come out at
# the size set below rather than being silently rescaled. Get the exact number
# by putting \the\textwidth in the document and reading the log, then set it
# here once.
FULL_W = 6.5      # TMLR \textwidth
HALF_W = 3.15     # two figures side by side, with a gutter
COL_W = HALF_W    # backwards-compatible alias


def save_fig(fig, name, formats=("pdf", "png")):
    """Write a figure to FIG_DIR in every requested format."""
    for ext in formats:
        fig.savefig(FIG_DIR / f"{name}.{ext}")
    print(f"saved {name}.{{{','.join(formats)}}} -> {FIG_DIR}")


def save_table(df, name, **kwargs):
    """Write a frame as CSV and as a booktabs LaTeX fragment."""
    df.to_csv(TABLE_DIR / f"{name}.csv")
    kwargs.setdefault("escape", False)
    kwargs.setdefault("index", False)
    (TABLE_DIR / f"{name}.tex").write_text(df.to_latex(**kwargs))
    print(f"saved {name}.{{csv,tex}} -> {TABLE_DIR}")

In [ ]:
# --- Experiment registry ----------------------------------------------------

TASK_ORDER = ("Cifar10", "Svhn", "Bikes", "Energy")

TASK_TYPE = {
    "Cifar10": "classification",
    "Svhn": "classification",
    "Bikes": "regression",
    "Energy": "regression",
}

TASK_LABEL = {
    "Cifar10": "CIFAR-10",
    "Svhn": "SVHN",
    "Bikes": "Bike Sharing",
    "Energy": "Energy",
}

SCORE_LABEL = {
    "classification": "validation accuracy",
    "regression": r"validation $R^2$",
}

# Keys inside *_metrics.json that carry the task metric.
SCORE_KEY = {
    "classification": "accuracy_metric",
    "regression": "r2_metric",
}

# Config columns written into the metric filenames.
CONFIG_COLS = ["predictors", "delegators", "pwidth", "dwidth", "mixing", "ambiguity"]


@dataclass(frozen=True)
class PairedFactor:
    """The Pool.paired variable of a sweep: the thing a matched block varies."""

    column: str
    levels: tuple[str, ...]
    baseline: str
    label: str
    level_labels: dict[str, str] = field(default_factory=dict)

    @property
    def block_size(self) -> int:
        return len(self.levels)

    @property
    def contrasts(self) -> tuple[str, ...]:
        return tuple(v for v in self.levels if v != self.baseline)

    def pretty(self, level: str) -> str:
        return self.level_labels.get(level, level)

    def colour(self, level: str) -> str:
        palette = [OKABE_ITO["black"], OKABE_ITO["blue"], OKABE_ITO["vermillion"]]
        return palette[self.levels.index(level) % len(palette)]


EXPERIMENTS = {
    "exp_aggregation": PairedFactor(
        column="mixing",
        levels=("sum", "product"),
        baseline="sum",
        label="Delegator aggregation",
        level_labels={"sum": "sum", "product": "product"},
    ),
    "exp_ambiguity_gradient": PairedFactor(
        column="ambiguity",
        levels=("none", "delegators", "both"),
        baseline="none",
        label="Ambiguity gradient",
        level_labels={
            "none": "no ambiguity gradient",
            "delegators": "delegators only",
            "both": "delegators + predictors",
        },
    ),
}

# Metrics carried through to the run-level tables.
TRACKED_METRICS = [
    "val_score",
    "train_score",
    "validation_loss",
    "validation_performance_loss",
    "validation_load_balancing_loss",
    "loss",
    "performance_loss",
]

LOSS_COL = "validation_performance_loss"   # used for early stopping
SCORE_COL = "val_score"                    # higher is better, task metric

## 1. Loading

Folder names look like

```
exp_ambiguity_gradient_d2f60c72f9_Cifar10_20260819_154050
<-- experiment ---------><launch><task-><-- timestamp -->
```

The experiment name contains underscores, so it is matched from the right
against the 10-hex-character launch id rather than split on `_`.

Each `*_metrics.json` inside a folder holds one run: a dict of
`(n_epochs, n_seeds)` arrays. The loader makes a single pass per file and keeps

* `curves`   - seed-averaged learning curves, thinned to `CURVE_POINTS` epochs,
  carrying the 2.5th/97.5th seed percentiles of the task metric,
* `seedvals` - per-seed values at the early-stopping epoch and the final epoch,
* `meta`     - one row per run: config, shapes, divergence bookkeeping.

Nothing ever holds `runs x epochs x seeds` rows at once, so the whole sweep fits
in a few tens of MB rather than a few GB.

In [ ]:
RUN_DIR_RE = re.compile(
    r"^(?P<experiment>.+)"
    r"_(?P<launch_id>[0-9a-f]{10})"
    r"_(?P<task>[A-Za-z0-9]+)"
    r"_(?P<date>\d{8})_(?P<time>\d{6})$"
)

# How many epochs to keep per run for curve plots. Selection epochs are found
# on the full-resolution arrays first, so thinning only affects the plots.
CURVE_POINTS = 400


def parse_case_stem(path: Path) -> dict:
    """`run_00007_predictors_16_..._ambiguity_none_metrics.json` -> dict."""
    parts = path.stem.removesuffix("_metrics").split("_")

    if len(parts) % 2 != 0:
        raise ValueError(f"Expected key_value pairs, got: {path.name}")

    out = {}
    for key, value in zip(parts[::2], parts[1::2], strict=True):
        try:
            out[key] = int(value)
        except ValueError:
            out[key] = value

    return out


def discover_runs(runs_dir: Path, experiments=None) -> pd.DataFrame:
    """One row per launch folder under `runs_dir`."""
    if not runs_dir.is_dir():
        raise FileNotFoundError(f"No runs directory at {runs_dir}")

    rows, skipped = [], []

    for folder in sorted(p for p in runs_dir.iterdir() if p.is_dir()):
        match = RUN_DIR_RE.match(folder.name)

        if match is None:
            skipped.append(folder.name)
            continue

        info = match.groupdict()

        if experiments is not None and info["experiment"] not in experiments:
            skipped.append(folder.name)
            continue

        if info["task"] not in TASK_TYPE:
            skipped.append(folder.name)
            continue

        rows.append(
            info
            | {
                "folder": folder,
                "task_type": TASK_TYPE[info["task"]],
                "n_cases": len(list(folder.glob("*_metrics.json"))),
                "started": pd.to_datetime(
                    info["date"] + info["time"], format="%Y%m%d%H%M%S"
                ),
            }
        )

    if skipped:
        print(f"skipped {len(skipped)} folder(s): {', '.join(sorted(skipped)[:6])}"
              + (" ..." if len(skipped) > 6 else ""))

    return pd.DataFrame(rows)

In [ ]:
def _thin(n_epochs: int, n_points: int) -> np.ndarray:
    """Indices spanning [0, n_epochs) with the endpoints always kept."""
    if n_epochs <= n_points:
        return np.arange(n_epochs)

    return np.unique(np.linspace(0, n_epochs - 1, n_points).round().astype(int))


def load_metrics_file(path: Path, launch: dict) -> tuple[dict, pd.DataFrame, pd.DataFrame]:
    """Read one run. Returns (meta_row, curve_rows, seed_rows)."""
    case = parse_case_stem(path)

    with path.open() as f:
        arrays = {k: np.asarray(v, dtype=np.float64) for k, v in json.load(f).items()}

    shapes = {k: v.shape for k, v in arrays.items()}

    if len(set(shapes.values())) != 1:
        raise ValueError(f"Metric shapes differ in {path.name}: {shapes}")

    shape = next(iter(shapes.values()))

    if len(shape) != 2:
        raise ValueError(
            f"Expected metrics shaped (epoch, seed), got {shape} in {path.name}"
        )

    n_epochs, n_seeds = shape

    # Harmonise the task metric so classification and regression share a column.
    score_key = SCORE_KEY[launch["task_type"]]

    if f"validation_{score_key}" not in arrays:
        raise KeyError(
            f"{path.name} has no validation_{score_key}; keys: {sorted(arrays)}"
        )

    arrays["val_score"] = arrays[f"validation_{score_key}"]
    arrays["train_score"] = arrays[score_key]

    # --- selection epochs ---------------------------------------------------
    # An epoch is eligible only if every seed is finite there, so early stopping
    # can never pick an epoch that only looks good because a seed blew up.
    vpl = arrays[LOSS_COL]
    eligible = np.isfinite(vpl).all(axis=1)

    if eligible.any():
        curve = np.where(eligible, vpl.mean(axis=1), np.inf)
        early_epoch = int(np.argmin(curve))
        last_finite_epoch = int(np.flatnonzero(eligible)[-1])
    else:
        early_epoch = last_finite_epoch = -1

    final_epoch = n_epochs - 1

    vloss = arrays["validation_loss"]
    nan_epochs = np.flatnonzero(~np.isfinite(vloss).all(axis=1))
    first_nan_epoch = int(nan_epochs[0]) if nan_epochs.size else -1

    ident = {
        "experiment": launch["experiment"],
        "launch_id": launch["launch_id"],
        "task": launch["task"],
        "task_type": launch["task_type"],
        "run_id": case["run"],
    }
    config = {col: case.get(col, pd.NA) for col in CONFIG_COLS}

    meta = ident | config | {
        "n_epochs": n_epochs,
        "n_seeds": n_seeds,
        "diverged": first_nan_epoch >= 0,
        "first_nan_epoch": first_nan_epoch,
        "first_nan_frac": (
            first_nan_epoch / max(n_epochs - 1, 1) if first_nan_epoch >= 0 else np.nan
        ),
        "early_epoch": early_epoch,
        "final_epoch": final_epoch,
        "last_finite_epoch": last_finite_epoch,
        "score_metric": score_key,
        "file": path.name,
    }

    # --- thinned, seed-averaged curves --------------------------------------
    keep = _thin(n_epochs, CURVE_POINTS)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)  # all-NaN slices
        curve_data = {
            name: np.nanmean(values[keep], axis=1).astype(np.float32)
            for name, values in arrays.items()
            if name in TRACKED_METRICS
        }

    curves = pd.DataFrame(curve_data)
    curves["epoch"] = keep
    curves["epoch_frac"] = keep / max(n_epochs - 1, 1)
    curves["finite_seed_frac"] = np.isfinite(vloss[keep]).mean(axis=1).astype(np.float32)

    # Seed spread of the task metric, so learning-curve figures can show a band
    # without keeping every seed of every epoch around.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        lo, hi = np.nanquantile(arrays["val_score"][keep], [0.025, 0.975], axis=1)

    curves["val_score_lo"] = lo.astype(np.float32)
    curves["val_score_hi"] = hi.astype(np.float32)

    for key, value in (ident | config).items():
        curves[key] = value

    # --- per-seed values at the selection epochs ----------------------------
    seed_frames = []

    for selection, epoch in (("early", early_epoch), ("final", final_epoch)):
        if epoch < 0:
            continue

        frame = pd.DataFrame(
            {
                name: values[epoch].astype(np.float32)
                for name, values in arrays.items()
                if name in TRACKED_METRICS
            }
        )
        frame["seed_id"] = np.arange(n_seeds)
        frame["selection"] = selection
        frame["epoch"] = epoch

        for key, value in (ident | config).items():
            frame[key] = value

        seed_frames.append(frame)

    seedvals = (
        pd.concat(seed_frames, ignore_index=True) if seed_frames else pd.DataFrame()
    )

    return meta, curves, seedvals


def load_runs(folders: pd.DataFrame):
    """Load every launch folder in `folders`."""
    metas, curve_parts, seed_parts = [], [], []

    for launch in folders.to_dict("records"):
        files = sorted(launch["folder"].glob("*_metrics.json"))

        for path in files:
            meta, curves, seedvals = load_metrics_file(path, launch)
            metas.append(meta)
            curve_parts.append(curves)

            if not seedvals.empty:
                seed_parts.append(seedvals)

        print(f"  {launch['folder'].name}: {len(files)} run(s)")

    meta = pd.DataFrame(metas)
    curves = pd.concat(curve_parts, ignore_index=True)
    seedvals = pd.concat(seed_parts, ignore_index=True)

    for frame in (meta, curves, seedvals):
        for col in ("experiment", "task", "task_type", "mixing", "ambiguity"):
            if col in frame:
                frame[col] = frame[col].astype("string")

    return meta, curves, seedvals

In [ ]:
folders = discover_runs(RUNS_DIR, experiments=set(EXPERIMENTS))

if folders.empty:
    raise FileNotFoundError(
        f"No recognised run folders under {RUNS_DIR}.\n"
        "Point the notebook elsewhere with LIQUID_RUNS_DIR=/path/to/runs."
    )

folders[["experiment", "task", "launch_id", "started", "n_cases"]].sort_values(
    ["experiment", "task"]
).reset_index(drop=True)

In [ ]:
meta, curves, seedvals = load_runs(folders)

print()
print(f"runs     : {len(meta)}")
print(f"curves   : {len(curves):,} rows  "
      f"({curves.memory_usage(deep=True).sum() / 1e6:.1f} MB)")
print(f"seedvals : {len(seedvals):,} rows  "
      f"({seedvals.memory_usage(deep=True).sum() / 1e6:.1f} MB)")

## 2. Provenance and integrity

Before any statistics: what actually landed on disk. This is the table to look
at when a sweep was resumed on a different cluster, since a resumed launch keeps
its original folder name and simply gains files.

In [ ]:
inventory = (
    meta.groupby(["experiment", "task"], dropna=False)
    .agg(
        launches=("launch_id", "nunique"),
        runs=("run_id", "size"),
        run_id_min=("run_id", "min"),
        run_id_max=("run_id", "max"),
        epochs=("n_epochs", "max"),
        seeds=("n_seeds", "max"),
        diverged=("diverged", "sum"),
        divergence_rate=("diverged", "mean"),
    )
    .reset_index()
)

inventory["complete"] = (
    inventory["runs"] == inventory["run_id_max"] - inventory["run_id_min"] + 1
)

inventory

In [ ]:
# Structural checks. Anything printed here invalidates the analysis below.
problems = []

dupes = meta.duplicated(["experiment", "task", "launch_id", "run_id"]).sum()
if dupes:
    problems.append(f"{dupes} duplicated (launch, run_id) rows")

for (exp, task), group in meta.groupby(["experiment", "task"], dropna=False):
    expected = set(range(group["run_id"].min(), group["run_id"].max() + 1))
    missing = sorted(expected - set(group["run_id"]))
    if missing:
        problems.append(
            f"{exp}/{task}: {len(missing)} missing run_id(s), "
            f"e.g. {missing[:8]}"
        )

    if group["n_seeds"].nunique() > 1:
        problems.append(f"{exp}/{task}: mixed seed counts {sorted(group['n_seeds'].unique())}")

    if group["n_epochs"].nunique() > 1:
        problems.append(
            f"{exp}/{task}: mixed epoch counts {sorted(group['n_epochs'].unique())} "
            "(runs stopped early?)"
        )

for exp, factor in EXPERIMENTS.items():
    sub = meta[meta["experiment"] == exp]
    if sub.empty:
        continue
    seen = set(sub[factor.column].dropna().unique())
    if seen != set(factor.levels):
        problems.append(
            f"{exp}: {factor.column} levels on disk {sorted(seen)} "
            f"!= expected {sorted(factor.levels)}"
        )

print("\n".join(problems) if problems else "no structural problems found")

## 3. Run-level summaries

Two selection rules, reported side by side throughout:

* **early** - the epoch minimising the seed-averaged
  `validation_performance_loss`, restricted to epochs where every seed is
  finite. This is the number a practitioner would get with early stopping.
* **final** - the last epoch of the budget. A run that diverged has no finite
  value here, which is the honest answer rather than a missing one.

Seeds are aggregated with a mean plus a standard error, so within-run noise
stays visible and is never confused with between-configuration spread. Note
that `select_last` / `select_early_stopping` in the exploratory notebook take
`tail(1)` and `idxmin()` over a frame that still has one row per seed, so they
collapse to a single arbitrary seed; that is what the aggregation below
replaces.

In [ ]:
ID_COLS = ["experiment", "task", "task_type", "launch_id", "run_id"]

agg_spec = {}
for metric in TRACKED_METRICS:
    agg_spec[f"{metric}_mean"] = (metric, "mean")
    agg_spec[f"{metric}_sem"] = (metric, lambda s: s.std(ddof=1) / np.sqrt(s.notna().sum()))

run_level = (
    seedvals.groupby(ID_COLS + CONFIG_COLS + ["selection", "epoch"], dropna=False)
    .agg(n_seeds=("seed_id", "size"), n_finite=(SCORE_COL, "count"), **agg_spec)
    .reset_index()
    .merge(
        meta[ID_COLS + ["diverged", "first_nan_frac", "n_epochs"]],
        on=ID_COLS,
        how="left",
    )
)

print(f"{len(run_level):,} rows ({run_level['selection'].nunique()} selections "
      f"x {len(meta)} runs)")

run_level.head()

## 4. Matched-block contrasts

`Experiment.cases` draws one random configuration and one PRNG key per block and
then emits `len(pool)` consecutive runs that differ **only** in the paired
factor. Blocks are therefore `(run_id - 1) // block_size`, and every block is
checked before use: right size, exactly the expected factor levels, and every
other configuration column constant. Blocks that fail are dropped and counted,
which is what makes a resumed or truncated sweep safe to analyse.

In [ ]:
def build_blocks(meta: pd.DataFrame, factor: PairedFactor) -> pd.DataFrame:
    """Assign a block id to each run and drop blocks that are not well formed."""
    held_constant = [c for c in CONFIG_COLS if c != factor.column]

    frame = meta.copy()
    frame["block_id"] = (frame["run_id"] - 1) // factor.block_size

    keys = ["experiment", "task", "launch_id", "block_id"]
    reasons = {"incomplete": 0, "wrong_levels": 0, "inconsistent_config": 0}
    good = []

    for key, block in frame.groupby(keys, dropna=False):
        if len(block) != factor.block_size:
            reasons["incomplete"] += 1
            continue

        if sorted(block[factor.column].dropna()) != sorted(factor.levels):
            reasons["wrong_levels"] += 1
            continue

        if block[held_constant].nunique(dropna=False).gt(1).any():
            reasons["inconsistent_config"] += 1
            continue

        good.append(block)

    dropped = sum(reasons.values())
    if dropped:
        print(f"  dropped {dropped} block(s): "
              + ", ".join(f"{k}={v}" for k, v in reasons.items() if v))

    if not good:
        return frame.iloc[0:0].assign(block_id=pd.Series(dtype=int))

    return pd.concat(good, ignore_index=True)


blocks = {}

for exp, factor in EXPERIMENTS.items():
    sub = meta[meta["experiment"] == exp]

    if sub.empty:
        print(f"{exp}: no runs on disk, skipping")
        continue

    print(f"{exp}: {len(sub)} runs, block size {factor.block_size}")
    kept = build_blocks(sub, factor)
    blocks[exp] = kept[["experiment", "task", "launch_id", "run_id", "block_id"]]

    print(f"  kept {kept.groupby(['task']).ngroups} task(s), "
          f"{len(kept) // factor.block_size} usable block(s)")

block_map = (
    pd.concat(blocks.values(), ignore_index=True)
    if blocks
    else pd.DataFrame(columns=["experiment", "task", "launch_id", "run_id", "block_id"])
)

paired = run_level.merge(
    block_map, on=["experiment", "task", "launch_id", "run_id"], how="inner"
)

print(f"\n{len(paired):,} run-level rows inside valid blocks")

In [ ]:
def paired_matrix(paired, experiment, task, selection, value_col, factor):
    """Blocks x factor levels matrix of one value column."""
    sub = paired[
        (paired["experiment"] == experiment)
        & (paired["task"] == task)
        & (paired["selection"] == selection)
    ]

    if sub.empty:
        return pd.DataFrame(columns=list(factor.levels))

    wide = sub.pivot_table(
        index=["launch_id", "block_id"],
        columns=factor.column,
        values=value_col,
        aggfunc="first",
        dropna=False,
        observed=True,
    )

    return wide.reindex(columns=list(factor.levels))


def block_config(paired, experiment, task, selection, factor):
    """The configuration held constant inside each block, indexed like
    `paired_matrix`. Kept separate rather than joined onto the matrix because
    the ambiguity level `delegators` collides with the config column of the
    same name; align the two by index instead."""
    held = [c for c in CONFIG_COLS if c != factor.column]

    sub = paired[
        (paired["experiment"] == experiment)
        & (paired["task"] == task)
        & (paired["selection"] == selection)
    ]

    return sub.groupby(["launch_id", "block_id"], dropna=False)[held].first()

In [ ]:
def bootstrap_ci(values, statistic=np.mean, n_boot=10_000, alpha=0.05, rng=RNG):
    """Percentile bootstrap CI of `statistic` over a 1-D sample."""
    values = np.asarray(values, dtype=float)

    if values.size < 2:
        return np.nan, np.nan

    idx = rng.integers(0, values.size, size=(n_boot, values.size))
    draws = statistic(values[idx], axis=1)

    return tuple(np.quantile(draws, [alpha / 2, 1 - alpha / 2]))


def rank_biserial(diff):
    """Matched-pairs rank-biserial correlation: -1 .. +1, 0 = no effect."""
    diff = np.asarray(diff, dtype=float)
    nonzero = diff[diff != 0]

    if nonzero.size == 0:
        return 0.0

    ranks = stats.rankdata(np.abs(nonzero))
    total = ranks.sum()

    return float((ranks[nonzero > 0].sum() - ranks[nonzero < 0].sum()) / total)


def paired_test(baseline, treatment, higher_is_better=True, n_boot=10_000):
    """Compare two matched vectors. `treatment - baseline`, NaN pairs dropped."""
    baseline = np.asarray(baseline, dtype=float)
    treatment = np.asarray(treatment, dtype=float)

    both_finite = np.isfinite(baseline) & np.isfinite(treatment)
    n_pairs = int(both_finite.sum())

    out = {
        "n_blocks": int(baseline.size),
        "n_pairs": n_pairs,
        "n_dropped": int(baseline.size - n_pairs),
        "baseline_mean": float(np.nanmean(baseline)) if np.isfinite(baseline).any() else np.nan,
        "treatment_mean": float(np.nanmean(treatment)) if np.isfinite(treatment).any() else np.nan,
    }

    if n_pairs < 3:
        return out | {
            k: np.nan
            for k in ("mean_diff", "ci_lo", "ci_hi", "median_diff", "win_rate",
                      "p_value", "effect_size")
        }

    diff = treatment[both_finite] - baseline[both_finite]
    ci_lo, ci_hi = bootstrap_ci(diff, n_boot=n_boot)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            p_value = float(stats.wilcoxon(diff, zero_method="wilcox").pvalue)
        except ValueError:      # every difference is exactly zero
            p_value = 1.0

    wins = diff > 0 if higher_is_better else diff < 0

    return out | {
        "mean_diff": float(diff.mean()),
        "ci_lo": float(ci_lo),
        "ci_hi": float(ci_hi),
        "median_diff": float(np.median(diff)),
        "win_rate": float(wins.mean()),
        "p_value": p_value,
        "effect_size": rank_biserial(diff if higher_is_better else -diff),
    }


def holm(p_values):
    """Holm-Bonferroni adjusted p-values, NaNs passed through."""
    p = np.asarray(p_values, dtype=float)
    out = np.full(p.shape, np.nan)
    finite = np.flatnonzero(np.isfinite(p))

    if finite.size == 0:
        return out

    order = finite[np.argsort(p[finite])]
    n = order.size
    running = 0.0

    for rank, idx in enumerate(order):
        running = max(running, (n - rank) * p[idx])
        out[idx] = min(running, 1.0)

    return out

In [ ]:
def contrast_table(paired, value_col, higher_is_better, n_boot=10_000):
    """Every (experiment, task, selection, contrast) comparison for one metric."""
    rows = []

    for experiment, factor in EXPERIMENTS.items():
        for task in TASK_ORDER:
            for selection in ("early", "final"):
                wide = paired_matrix(
                    paired, experiment, task, selection, value_col, factor
                )

                if wide.empty:
                    continue

                for level in factor.contrasts:
                    result = paired_test(
                        wide[factor.baseline].to_numpy(),
                        wide[level].to_numpy(),
                        higher_is_better=higher_is_better,
                        n_boot=n_boot,
                    )

                    rows.append(
                        {
                            "experiment": experiment,
                            "factor": factor.column,
                            "task": task,
                            "selection": selection,
                            "contrast": f"{level} - {factor.baseline}",
                            "level": level,
                            "baseline": factor.baseline,
                            "metric": value_col,
                        }
                        | result
                    )

    table = pd.DataFrame(rows)

    if table.empty:
        return table

    # Correct within each experiment x selection family.
    table["p_holm"] = np.nan
    for _, idx in table.groupby(["experiment", "selection"], dropna=False).groups.items():
        table.loc[idx, "p_holm"] = holm(table.loc[idx, "p_value"])

    return table


score_contrasts = contrast_table(paired, f"{SCORE_COL}_mean", higher_is_better=True)
loss_contrasts = contrast_table(paired, f"{LOSS_COL}_mean", higher_is_better=False)

score_contrasts[
    ["experiment", "task", "selection", "contrast", "n_pairs", "n_dropped",
     "mean_diff", "ci_lo", "ci_hi", "win_rate", "effect_size", "p_holm"]
].round(4)

### Sensitivity to divergence

`n_dropped` above counts blocks where at least one variant produced no finite
value. Dropping them assumes divergence is unrelated to the variant, which is
exactly what is in question when one aggregation rule blows up more often than
the other. The check below re-runs the comparison with the diverged member of a
block replaced by the worst finite score seen for that task, i.e. the most
pessimistic reading. If the sign of the effect survives both readings the
conclusion does not rest on the exclusion.

In [ ]:
def worst_case_table(paired, value_col=f"{SCORE_COL}_mean", n_boot=5_000):
    rows = []

    for experiment, factor in EXPERIMENTS.items():
        for task in TASK_ORDER:
            for selection in ("early", "final"):
                wide = paired_matrix(paired, experiment, task, selection, value_col, factor)

                if wide.empty or not np.isfinite(wide.to_numpy()).any():
                    continue

                floor = np.nanmin(wide.to_numpy())
                filled = wide.fillna(floor)

                for level in factor.contrasts:
                    result = paired_test(
                        filled[factor.baseline].to_numpy(),
                        filled[level].to_numpy(),
                        higher_is_better=True,
                        n_boot=n_boot,
                    )

                    rows.append(
                        {
                            "experiment": experiment,
                            "task": task,
                            "selection": selection,
                            "contrast": f"{level} - {factor.baseline}",
                            "imputed_floor": float(floor),
                        }
                        | result
                    )

    return pd.DataFrame(rows)


worst_case = worst_case_table(paired)

if not worst_case.empty:
    comparison = score_contrasts.merge(
        worst_case,
        on=["experiment", "task", "selection", "contrast"],
        suffixes=("_complete", "_worstcase"),
    )
    comparison["sign_flips"] = (
        np.sign(comparison["mean_diff_complete"])
        != np.sign(comparison["mean_diff_worstcase"])
    )
    display_cols = [
        "experiment", "task", "selection", "contrast",
        "n_dropped_complete", "mean_diff_complete", "mean_diff_worstcase",
        "win_rate_complete", "win_rate_worstcase", "sign_flips",
    ]
    comparison = comparison[display_cols].round(4)
else:
    comparison = pd.DataFrame()

comparison

## 5. Figures

Everything below writes to `paper_plots/` as both PDF (for the paper) and PNG
(for slides and quick viewing). Figure widths are set to a two-column layout:
`FULL_W` for a figure spanning the text block, `HALF_W` for two side by side.

TMLR is single-column, so `FULL_W` is the template's `\textwidth`. Set it to the
real value once (section 0) and every figure below inherits it; including a
figure at `[width=\textwidth]` then reproduces it at 1:1, which is what keeps
the tick labels at the 8pt they were drawn at instead of whatever a rescale
turns them into.

In [ ]:
AVAILABLE = [e for e in EXPERIMENTS if (meta["experiment"] == e).any()]


def tasks_for(experiment):
    present = set(meta.loc[meta["experiment"] == experiment, "task"])
    return [t for t in TASK_ORDER if t in present]


def significance_stars(p):
    if not np.isfinite(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return ""


def level_legend(fig, factor, levels, title=None):
    """One shared legend under the figure instead of one per axes."""
    handles = [
        Line2D([], [], color=factor.colour(level), marker="o", markersize=4,
               linewidth=1.4, label=factor.pretty(level))
        for level in levels
    ]
    fig.legend(
        handles=handles,
        loc="outside lower center",
        ncols=len(handles),
        title=title,
    )


def power_of_two_axis(ax, values):
    """Label a symlog-2 axis with the actual counts, not 2^k."""
    values = sorted({int(v) for v in values})
    ax.set_xscale("symlog", base=2, linthresh=1.9)
    ax.xaxis.set_major_locator(mpl.ticker.FixedLocator(values))
    ax.xaxis.set_minor_locator(mpl.ticker.NullLocator())
    ax.set_xticklabels([str(v) for v in values])


print("experiments on disk:", ", ".join(AVAILABLE) or "none")
for exp in AVAILABLE:
    print(f"  {exp}: {', '.join(tasks_for(exp))}")

### Figure 1 - paired effect sizes

The headline figure. Each marker is the mean paired difference in the task
metric between a variant and its baseline, over matched blocks that share a
configuration and a PRNG key; whiskers are 10,000-sample percentile bootstrap
intervals. An interval clear of the dashed zero line is an effect that survives
resampling. Stars are Holm-corrected Wilcoxon signed-rank p-values
(`*` < 0.05, `**` < 0.01, `***` < 0.001).

The two panels share an x axis on purpose: early stopping and the end of the
budget are two readings of the same experiment, and putting them on separate
scales would make a large late effect look like a small early one.

In [ ]:
def forest_plot(table, experiment, metric_label="task metric", fname=None):
    factor = EXPERIMENTS[experiment]
    sub = table[table["experiment"] == experiment]

    if sub.empty:
        print(f"{experiment}: nothing to plot")
        return None

    tasks = [t for t in TASK_ORDER if t in set(sub["task"])]
    selections = ["early", "final"]
    contrasts = list(factor.contrasts)

    fig, axes = plt.subplots(
        1, len(selections),
        figsize=(FULL_W, 0.28 * len(tasks) * max(len(contrasts), 1) + 1.15),
        sharey=True,
        sharex=True,
    )
    axes = np.atleast_1d(axes)

    offsets = np.linspace(0.2, -0.2, len(contrasts)) if len(contrasts) > 1 else [0.0]
    y_of = {task: i for i, task in enumerate(reversed(tasks))}

    for ax, selection in zip(axes, selections):
        part = sub[sub["selection"] == selection]

        for boundary in range(len(tasks) - 1):
            ax.axhline(boundary + 0.5, color="0.92", linewidth=0.7, zorder=0)

        for offset, level in zip(offsets, contrasts):
            colour = factor.colour(level)

            for _, row in part[part["level"] == level].iterrows():
                if not np.isfinite(row["mean_diff"]):
                    continue

                y = y_of[row["task"]] + offset

                ax.plot([row["ci_lo"], row["ci_hi"]], [y, y],
                        color=colour, linewidth=1.4, solid_capstyle="round")
                ax.plot(row["mean_diff"], y, marker="o", color=colour,
                        markersize=4.5, markeredgecolor="white",
                        markeredgewidth=0.6, zorder=3)

                stars = significance_stars(row["p_holm"])
                if stars:
                    ax.annotate(stars, (row["ci_hi"], y), textcoords="offset points",
                                xytext=(4, -1), fontsize=7, color=colour, va="center")

        ax.axvline(0, color="0.3", linestyle="--", linewidth=0.9, zorder=1)
        ax.set_title("early stopping" if selection == "early" else "end of budget")
        ax.set_xlabel(f"$\\Delta$ {metric_label}")
        ax.set_yticks(range(len(tasks)))
        ax.set_yticklabels([TASK_LABEL[t] for t in reversed(tasks)])
        ax.set_ylim(-0.6, len(tasks) - 0.4)
        ax.grid(axis="y", visible=False)

    level_legend(fig, factor, contrasts, title=f"vs {factor.pretty(factor.baseline)}")

    fig.suptitle(f"{factor.label}: paired effect on {metric_label}")

    if fname:
        save_fig(fig, fname)

    return fig


for exp in AVAILABLE:
    forest_plot(score_contrasts, exp, fname=f"fig1_forest_{exp}")

plt.show()

### Figure 2 - block-level scatter

The forest plot compresses each task to one number. This shows every block:
baseline on the x axis, variant on the y axis, one panel per task. Points above
the diagonal favour the variant. Marker colour encodes the number of delegators,
which is the axis along which the effect turns out to move.

Axis limits start at the 2nd percentile because $R^2$ is unbounded below and a
single collapsed run would otherwise squash every informative point into a
corner. The blocks that fall outside are counted in the annotation.

In [ ]:
def paired_scatter(paired, experiment, selection="early", value_col=f"{SCORE_COL}_mean",
                   fname=None):
    factor = EXPERIMENTS[experiment]
    tasks = tasks_for(experiment)
    contrasts = list(factor.contrasts)

    if not tasks:
        return None

    n_rows = len(contrasts)
    fig, axes = plt.subplots(
        n_rows, len(tasks),
        figsize=(FULL_W, 1.5 * n_rows + 0.95),
        squeeze=False,
    )

    norm = mpl.colors.LogNorm(vmin=1, vmax=32)
    scatter = None

    for row, level in enumerate(contrasts):
        for col, task in enumerate(tasks):
            ax = axes[row][col]
            wide = paired_matrix(paired, experiment, task, selection, value_col, factor)
            cfg = block_config(paired, experiment, task, selection, factor)

            if wide.empty:
                ax.set_axis_off()
                continue

            frame = pd.DataFrame(
                {
                    "baseline": wide[factor.baseline],
                    "treatment": wide[level],
                    "delegators": cfg["delegators"].reindex(wide.index),
                }
            ).replace([np.inf, -np.inf], np.nan).dropna()

            if frame.empty:
                ax.set_axis_off()
                continue

            x = frame["baseline"].to_numpy()
            y = frame["treatment"].to_numpy()

            # Robust floor: only genuine outliers are pushed off-scale, so a
            # well behaved panel keeps every point.
            both = np.concatenate([x, y])
            q05, q95 = np.quantile(both, [0.05, 0.95])
            spread = max(q95 - q05, 1e-9)
            lo = max(both.min(), q05 - 1.5 * spread)
            hi = both.max()
            pad = 0.05 * max(hi - lo, 1e-9)
            lo, hi = lo - pad, hi + pad
            clipped = int(((x < lo) | (y < lo)).sum())

            scatter = ax.scatter(
                x, y, c=frame["delegators"].astype(float),
                norm=norm, cmap="viridis",
                s=13, alpha=0.85, linewidths=0.3, edgecolors="white",
            )
            ax.plot([lo, hi], [lo, hi], "--", color="0.3", linewidth=0.9, zorder=0)
            ax.set_xlim(lo, hi)
            ax.set_ylim(lo, hi)
            ax.set_aspect("equal", adjustable="box")
            ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(3))
            ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(3))

            note = f"{(y > x).mean():.0%} above"
            if clipped:
                note += f"\n{clipped} off-scale"
            ax.annotate(note, (0.05, 0.95), xycoords="axes fraction",
                        fontsize=6.5, va="top")

            if row == 0:
                ax.set_title(TASK_LABEL[task])
            if col == 0:
                ax.set_ylabel(factor.pretty(level), fontsize=8)

    if scatter is not None:
        cbar = fig.colorbar(scatter, ax=axes, shrink=0.55, aspect=22, pad=0.012)
        cbar.set_label("delegators")
        cbar.set_ticks([1, 2, 4, 8, 16, 32])
        cbar.ax.set_yticklabels(["1", "2", "4", "8", "16", "32"])
        cbar.ax.minorticks_off()

    fig.supxlabel(factor.pretty(factor.baseline), fontsize=9)
    fig.suptitle(
        f"{factor.label}: matched blocks, "
        f"{'early stopping' if selection == 'early' else 'end of budget'}"
    )

    if fname:
        save_fig(fig, fname)

    return fig


for exp in AVAILABLE:
    paired_scatter(paired, exp, selection="early", fname=f"fig2_scatter_{exp}")

plt.show()

### Figure 3 - distribution of paired differences

The exploratory notebook plotted a histogram of relative differences per task.
This keeps the idea but shows the whole distribution, marks the median and the
bootstrap interval of the mean, and prints how many blocks each violin is built
from, so a thin task cannot be mistaken for a solid one.

Classification and regression get separate rows: an accuracy difference lives on
a scale two orders of magnitude smaller than an $R^2$ difference, and sharing an
axis makes the classification violins invisible.

In [ ]:
def difference_violins(paired, experiment, selection="early",
                       value_col=f"{SCORE_COL}_mean", fname=None):
    factor = EXPERIMENTS[experiment]
    tasks = tasks_for(experiment)
    contrasts = list(factor.contrasts)

    if not tasks:
        return None

    task_types = [
        t for t in ("classification", "regression")
        if any(TASK_TYPE[task] == t for task in tasks)
    ]

    fig, axes = plt.subplots(
        len(task_types), len(contrasts),
        figsize=(min(FULL_W, 2.6 * len(contrasts) + 0.7), 1.9 * len(task_types) + 0.7),
        squeeze=False,
    )

    for row, task_type in enumerate(task_types):
        for col, level in enumerate(contrasts):
            ax = axes[row][col]
            data, labels, counts = [], [], []

            for task in tasks:
                if TASK_TYPE[task] != task_type:
                    continue

                wide = paired_matrix(paired, experiment, task, selection, value_col, factor)

                if wide.empty:
                    continue

                diff = (
                    (wide[level] - wide[factor.baseline])
                    .replace([np.inf, -np.inf], np.nan)
                    .dropna()
                )

                if diff.empty:
                    continue

                data.append(diff.to_numpy())
                labels.append(TASK_LABEL[task])
                counts.append(len(diff))

            if not data:
                ax.set_axis_off()
                continue

            positions = np.arange(len(data))
            parts = ax.violinplot(data, positions=positions, widths=0.75,
                                  showextrema=False, showmedians=False)

            for body in parts["bodies"]:
                body.set_facecolor(factor.colour(level))
                body.set_alpha(0.35)
                body.set_edgecolor("none")

            for pos, values in zip(positions, data):
                ci = bootstrap_ci(values, n_boot=5_000)
                ax.plot([pos, pos], ci, color=factor.colour(level), linewidth=1.6,
                        solid_capstyle="round", zorder=3)
                ax.plot(pos, np.median(values), marker="o", color="white",
                        markeredgecolor=factor.colour(level), markeredgewidth=1.1,
                        markersize=4.5, zorder=4)

            ax.axhline(0, color="0.3", linestyle="--", linewidth=0.9, zorder=0)
            ax.set_xticks(positions)
            ax.set_xticklabels([f"{lab}\n$n$={n}" for lab, n in zip(labels, counts)])
            ax.set_xlim(-0.7, len(data) - 0.3)
            ax.grid(axis="x", visible=False)

            if row == 0:
                ax.set_title(factor.pretty(level))
            if col == 0:
                ax.set_ylabel(f"$\\Delta$ {SCORE_LABEL[task_type]}".replace("validation ", ""))

    fig.suptitle(
        f"{factor.label}: paired differences per block, "
        f"vs {factor.pretty(factor.baseline)}"
    )

    if fname:
        save_fig(fig, fname)

    return fig


for exp in AVAILABLE:
    difference_violins(paired, exp, fname=f"fig3_differences_{exp}")

plt.show()

### Figure 4 - divergence

The exploratory pass found that `product` mixing produces NaN losses. Divergence
is a result in its own right, so it gets a figure: how often each variant fails,
and where in configuration space it fails.

Only levels that actually produce at least one NaN get a heatmap panel, so the
figure stays honest about which variants are stable.

In [ ]:
def divergence_figure(meta, experiment, fname=None):
    factor = EXPERIMENTS[experiment]
    sub = meta[meta["experiment"] == experiment]
    tasks = tasks_for(experiment)

    if sub.empty or not tasks:
        return None

    diverging_levels = [
        level for level in factor.levels
        if sub.loc[sub[factor.column] == level, "diverged"].any()
    ]

    if not diverging_levels:
        print(f"{experiment}: no divergence anywhere, skipping heatmaps")
        diverging_levels = []

    n_bottom = len(diverging_levels)
    height = 2.0 + (2.1 if n_bottom else 0.0)

    fig = plt.figure(figsize=(FULL_W, height))
    gs = fig.add_gridspec(
        2 if n_bottom else 1,
        max(n_bottom, 1),
        height_ratios=[1.0, 1.15] if n_bottom else [1.0],
    )

    # --- (a) divergence rate per task and level -----------------------------
    ax = fig.add_subplot(gs[0, :])
    rates = (
        sub.groupby(["task", factor.column], dropna=False)["diverged"]
        .agg(rate="mean", n="size")
        .reset_index()
    )

    width = 0.8 / len(factor.levels)
    base = np.arange(len(tasks))

    for i, level in enumerate(factor.levels):
        part = rates[rates[factor.column] == level].set_index("task").reindex(tasks)
        offset = (i - (len(factor.levels) - 1) / 2) * width

        bars = ax.bar(
            base + offset, part["rate"].fillna(0.0), width=width * 0.92,
            color=factor.colour(level), label=factor.pretty(level),
            edgecolor="white", linewidth=0.5,
        )

        for bar, rate in zip(bars, part["rate"]):
            if np.isfinite(rate) and rate > 0:
                ax.annotate(
                    f"{rate:.0%}", (bar.get_x() + bar.get_width() / 2, rate),
                    textcoords="offset points", xytext=(0, 2),
                    ha="center", fontsize=6.5,
                )

    ax.set_xticks(base)
    ax.set_xticklabels([TASK_LABEL[t] for t in tasks])
    ax.set_ylabel("runs producing NaN")
    ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(xmax=1, decimals=0))
    ax.set_title("(a) divergence rate", loc="left")
    ax.legend(ncols=len(factor.levels), loc="upper right", fontsize=7)
    ax.grid(axis="x", visible=False)

    # --- (b..) where in configuration space ---------------------------------
    image = None

    for i, level in enumerate(diverging_levels):
        ax = fig.add_subplot(gs[1, i])
        part = sub[sub[factor.column] == level]

        grid = (
            part.pivot_table(
                index="delegators", columns="pwidth", values="diverged",
                aggfunc="mean", observed=True,
            )
            .sort_index()
            .sort_index(axis=1)
        )

        if grid.empty:
            ax.set_axis_off()
            continue

        values = grid.to_numpy(dtype=float)
        image = ax.imshow(values, cmap="magma_r", vmin=0, vmax=1,
                          aspect="auto", origin="lower")

        for yi in range(values.shape[0]):
            for xi in range(values.shape[1]):
                if np.isfinite(values[yi, xi]):
                    ax.text(xi, yi, f"{values[yi, xi]:.2f}", ha="center", va="center",
                            fontsize=6.5,
                            color="white" if values[yi, xi] > 0.55 else "0.15")

        ax.set_xticks(range(values.shape[1]), grid.columns.astype(int))
        ax.set_yticks(range(values.shape[0]), grid.index.astype(int))
        ax.set_xlabel("predictor width")
        ax.set_title(f"({chr(98 + i)}) {factor.pretty(level)}", loc="left")
        ax.grid(visible=False)

        if i == 0:
            ax.set_ylabel("delegators")

    if image is not None:
        fig.colorbar(image, ax=fig.axes[1:], shrink=0.9, pad=0.015, label="NaN rate")

    fig.suptitle(f"{factor.label}: training instability")

    if fname:
        save_fig(fig, fname)

    return fig


for exp in AVAILABLE:
    divergence_figure(meta, exp, fname=f"fig4a_divergence_{exp}")

plt.show()

The survival curve reads as "fraction of runs still producing a finite loss
after this fraction of the epoch budget". A curve that drops immediately means
runs that never trained; a curve that decays late means runs that trained and
then blew up. The distinction matters: the first is a bad initialisation, the
second is an optimisation problem that a schedule or a clip could fix.

In [ ]:
def survival_figure(meta, experiment, fname=None):
    factor = EXPERIMENTS[experiment]
    sub = meta[meta["experiment"] == experiment]
    tasks = tasks_for(experiment)

    if sub.empty or not tasks:
        return None

    fig, axes = plt.subplots(
        1, len(tasks), figsize=(FULL_W, 2.0), sharey=True, squeeze=False
    )
    grid = np.linspace(0, 1, 201)

    for ax, task in zip(axes[0], tasks):
        part = sub[sub["task"] == task]

        for level in factor.levels:
            runs = part[part[factor.column] == level]

            if runs.empty:
                continue

            # A run that never diverged has no failure fraction and survives.
            failure = runs["first_nan_frac"].to_numpy(dtype=float)
            alive = np.array([
                np.mean(~np.isfinite(failure) | (failure > t)) for t in grid
            ])

            ax.plot(grid, alive, color=factor.colour(level),
                    label=factor.pretty(level))

        ax.set_ylim(-0.03, 1.03)
        ax.set_xlim(0, 1)
        ax.set_title(TASK_LABEL[task])
        ax.set_xlabel("fraction of epoch budget")
        ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(xmax=1, decimals=0))

    axes[0][0].set_ylabel("runs still finite")
    level_legend(fig, factor, factor.levels)

    fig.suptitle(f"{factor.label}: when runs diverge")

    if fname:
        save_fig(fig, fname)

    return fig


for exp in AVAILABLE:
    survival_figure(meta, exp, fname=f"fig4b_survival_{exp}")

plt.show()

### Figure 5 - where the effect lives

A single averaged effect hides whether a variant helps everywhere or only in
part of the configuration space. This bins the matched blocks by a capacity axis
and plots the mean paired difference with a bootstrap band inside each bin, so a
trend along "more delegators" or "wider predictors" becomes visible. Bins with
fewer than three blocks are dropped rather than plotted as a point estimate of
one.

In [ ]:
def _binned_bootstrap(frame, x_col, y_col, min_count=3, n_boot=4_000):
    xs, means, los, his = [], [], [], []

    for value, group in frame.groupby(x_col, observed=True):
        if len(group) < min_count:
            continue

        sample = group[y_col].to_numpy()
        lo, hi = bootstrap_ci(sample, n_boot=n_boot)

        xs.append(float(value))
        means.append(sample.mean())
        los.append(lo)
        his.append(hi)

    order = np.argsort(xs)

    return (
        np.asarray(xs)[order],
        np.asarray(means)[order],
        np.asarray(los)[order],
        np.asarray(his)[order],
    )


def effect_vs_axis(paired, experiment, axis="delegators", selection="early",
                   value_col=f"{SCORE_COL}_mean", fname=None):
    factor = EXPERIMENTS[experiment]
    tasks = tasks_for(experiment)

    if not tasks:
        return None

    fig, axes = plt.subplots(
        1, len(tasks), figsize=(FULL_W, 2.1), squeeze=False
    )
    seen_values = set()

    for ax, task in zip(axes[0], tasks):
        wide = paired_matrix(paired, experiment, task, selection, value_col, factor)
        cfg = block_config(paired, experiment, task, selection, factor)

        if wide.empty:
            ax.set_axis_off()
            continue

        for level in factor.contrasts:
            part = pd.DataFrame(
                {
                    "diff": wide[level] - wide[factor.baseline],
                    axis: cfg[axis].reindex(wide.index),
                }
            ).replace([np.inf, -np.inf], np.nan).dropna()

            if part.empty:
                continue

            xs, means, los, his = _binned_bootstrap(part, axis, "diff")

            if xs.size == 0:
                continue

            seen_values.update(xs.astype(int).tolist())
            colour = factor.colour(level)
            ax.plot(xs, means, marker="o", color=colour, label=factor.pretty(level))
            ax.fill_between(xs, los, his, color=colour, alpha=0.18, linewidth=0)

        ax.axhline(0, color="0.3", linestyle="--", linewidth=0.9, zorder=0)
        ax.set_xlabel(axis)
        ax.set_title(TASK_LABEL[task])

    for ax in axes[0]:
        if ax.has_data() and seen_values:
            power_of_two_axis(ax, seen_values)

    axes[0][0].set_ylabel("$\\Delta$ task metric")
    level_legend(fig, factor, factor.contrasts,
                 title=f"vs {factor.pretty(factor.baseline)}")

    fig.suptitle(f"{factor.label}: paired effect vs {axis}")

    if fname:
        save_fig(fig, fname)

    return fig


for exp in AVAILABLE:
    effect_vs_axis(paired, exp, axis="delegators",
                   fname=f"fig5_effect_vs_delegators_{exp}")

plt.show()

### Figure 6 - absolute performance vs number of delegators

The companion to Figure 5: not the paired difference but the level itself, which
is what a reader wants when asking "how good does this get". Diverged runs
cannot be placed on an absolute axis, so they are excluded and counted in the
annotation; read this figure together with Figure 4.

In [ ]:
def performance_vs_delegators(paired, experiment, selection="early",
                              value_col=f"{SCORE_COL}_mean", fname=None):
    factor = EXPERIMENTS[experiment]
    tasks = tasks_for(experiment)

    if not tasks:
        return None

    fig, axes = plt.subplots(1, len(tasks), figsize=(FULL_W, 2.1), squeeze=False)
    seen_values = set()
    previous_type = None

    for ax, task in zip(axes[0], tasks):
        part = paired[
            (paired["experiment"] == experiment)
            & (paired["task"] == task)
            & (paired["selection"] == selection)
        ].replace([np.inf, -np.inf], np.nan)

        if part.empty:
            ax.set_axis_off()
            continue

        n_missing = int(part[value_col].isna().sum())

        for level in factor.levels:
            level_part = part[part[factor.column] == level].dropna(subset=[value_col])

            if level_part.empty:
                continue

            xs, means, los, his = _binned_bootstrap(level_part, "delegators", value_col)

            if xs.size == 0:
                continue

            seen_values.update(xs.astype(int).tolist())
            colour = factor.colour(level)
            ax.plot(xs, means, marker="o", color=colour, label=factor.pretty(level))
            ax.fill_between(xs, los, his, color=colour, alpha=0.18, linewidth=0)

        ax.set_xlabel("delegators")
        ax.set_title(TASK_LABEL[task])

        if n_missing:
            ax.annotate(f"{n_missing} diverged excluded", (0.04, 0.05),
                        xycoords="axes fraction", fontsize=6.5, color="0.35")

        # Label the y axis once per metric, since accuracy and R^2 alternate.
        if TASK_TYPE[task] != previous_type:
            ax.set_ylabel(SCORE_LABEL[TASK_TYPE[task]])
            previous_type = TASK_TYPE[task]

    for ax in axes[0]:
        if ax.has_data() and seen_values:
            power_of_two_axis(ax, seen_values)

    level_legend(fig, factor, factor.levels)

    fig.suptitle(
        f"{factor.label}: "
        f"{'early stopping' if selection == 'early' else 'end of budget'}"
    )

    if fname:
        save_fig(fig, fname)

    return fig


for exp in AVAILABLE:
    performance_vs_delegators(paired, exp, fname=f"fig6_metric_vs_delegators_{exp}")

plt.show()

### Figure 7 - learning curves

Validation curves for the highest-capacity matched block in each task: the solid
line is the seed mean, the band spans the 2.5th to 97.5th percentile across the
five seeds. This is the figure that shows *how* a variant differs - earlier
convergence, a different plateau, or a late collapse - rather than only by how
much.

The block is chosen by ranking the configurations actually present, not by
assuming `run_id` 1 is the largest, so the figure stays correct on a partially
completed or resumed sweep.

In [ ]:
def widest_block(paired, experiment, task, selection="early"):
    """The block_id with the highest capacity, lexicographically over the
    configuration columns."""
    part = paired[
        (paired["experiment"] == experiment)
        & (paired["task"] == task)
        & (paired["selection"] == selection)
    ]

    if part.empty:
        return None, None

    order = ["predictors", "delegators", "pwidth", "dwidth"]
    ranked = (
        part.groupby(["launch_id", "block_id"], dropna=False)[order]
        .first()
        .sort_values(order, ascending=False)
    )

    if ranked.empty:
        return None, None

    return ranked.index[0], ranked.iloc[0]


def learning_curves(curves, paired, experiment, metric=SCORE_COL, fname=None):
    factor = EXPERIMENTS[experiment]
    tasks = tasks_for(experiment)

    if not tasks:
        return None

    fig, axes = plt.subplots(1, len(tasks), figsize=(FULL_W, 2.1), squeeze=False)

    for ax, task in zip(axes[0], tasks):
        key, config = widest_block(paired, experiment, task)

        if key is None:
            ax.set_axis_off()
            continue

        launch_id, block_id = key
        run_ids = set(
            paired.loc[
                (paired["experiment"] == experiment)
                & (paired["launch_id"] == launch_id)
                & (paired["block_id"] == block_id),
                "run_id",
            ]
        )

        selected = curves[
            (curves["experiment"] == experiment)
            & (curves["launch_id"] == launch_id)
            & (curves["run_id"].isin(run_ids))
        ]

        for level in factor.levels:
            part = selected[selected[factor.column] == level].sort_values("epoch")

            if part.empty:
                continue

            colour = factor.colour(level)
            ax.plot(part["epoch"], part[metric], color=colour,
                    label=factor.pretty(level))

            if metric == SCORE_COL:
                ax.fill_between(
                    part["epoch"], part["val_score_lo"], part["val_score_hi"],
                    color=colour, alpha=0.15, linewidth=0,
                )

        ax.set_title(TASK_LABEL[task])
        ax.set_xlabel("epoch")
        ax.annotate(
            f"P={config['predictors']}, D={config['delegators']}\n"
            f"$w_p$={config['pwidth']}, $w_d$={config['dwidth']}",
            (0.96, 0.06), xycoords="axes fraction", ha="right",
            fontsize=6.5, color="0.35",
        )

        if ax is axes[0][0]:
            ax.set_ylabel("validation metric")

    level_legend(fig, factor, factor.levels)

    fig.suptitle(f"{factor.label}: highest-capacity block")

    if fname:
        save_fig(fig, fname)

    return fig


for exp in AVAILABLE:
    learning_curves(curves, paired, exp, fname=f"fig7_learning_curves_{exp}")

plt.show()

## 6. Tables for the paper

`paper_results.tex` is the main results table: one row per task and contrast,
with the paired mean difference and its bootstrap interval, the win rate over
blocks, the rank-biserial effect size and the Holm-corrected p-value.
`paper_divergence.tex` is the companion instability table. Both are written as
`booktabs` fragments, so `\input{}` them and keep the caption in the paper.

In [ ]:
def format_results(table):
    if table.empty:
        return table

    out = pd.DataFrame({
        "Experiment": table["experiment"].map(
            {k: EXPERIMENTS[k].label for k in EXPERIMENTS}
        ),
        "Task": table["task"].map(TASK_LABEL),
        "Selection": table["selection"].map(
            {"early": "early stop", "final": "end of budget"}
        ),
        "Contrast": table["contrast"],
        "$n$": table["n_pairs"].astype("Int64"),
        "Dropped": table["n_dropped"].astype("Int64"),
    })

    out[r"$\Delta$ metric [95\% CI]"] = [
        "--" if not np.isfinite(m) else f"{m:+.4f} [{lo:+.4f}, {hi:+.4f}]"
        for m, lo, hi in zip(table["mean_diff"], table["ci_lo"], table["ci_hi"])
    ]
    out["Win rate"] = [
        "--" if not np.isfinite(w) else f"{w:.0%}".replace("%", r"\%")
        for w in table["win_rate"]
    ]
    out["$r_{rb}$"] = [
        "--" if not np.isfinite(e) else f"{e:+.2f}" for e in table["effect_size"]
    ]
    out[r"$p_{\mathrm{Holm}}$"] = [
        "--" if not np.isfinite(p)
        else ("$<$0.001" if p < 0.001 else f"{p:.3f}")
        for p in table["p_holm"]
    ]

    return out


results_table = format_results(score_contrasts)

if not results_table.empty:
    save_table(results_table, "paper_results", column_format="llllrrlrrr")

results_table

In [ ]:
divergence_rows = []

for experiment, factor in EXPERIMENTS.items():
    sub = meta[meta["experiment"] == experiment]

    for task in tasks_for(experiment) if not sub.empty else []:
        part = sub[sub["task"] == task]

        row = {
            "Experiment": factor.label,
            "Task": TASK_LABEL[task],
            "Runs": len(part),
        }

        for level in factor.levels:
            level_part = part[part[factor.column] == level]
            row[f"{factor.pretty(level)} NaN rate"] = (
                level_part["diverged"].mean() if len(level_part) else np.nan
            )

        row["median failure epoch frac"] = part["first_nan_frac"].median()
        divergence_rows.append(row)

divergence_table = pd.DataFrame(divergence_rows)

if not divergence_table.empty:
    save_table(divergence_table.round(3), "paper_divergence")

divergence_table.round(3)

## 7. Notes and caveats

* **What the tests are on.** Every p-value is a Wilcoxon signed-rank test over
  matched blocks, paired within a configuration and a PRNG key. Blocks within a
  task are not independent of each other in every respect, so these are
  within-task statements, and Holm correction is applied within
  experiment x selection rather than across everything at once.
* **Seeds are not replicates of the contrast.** The five seeds inside a run
  share the block's configuration. They bound within-run noise and are reported
  as standard errors and as the band in Figure 7, but the unit of analysis for
  every contrast is the block.
* **`n_dropped` is not missing at random.** See the sensitivity table in
  section 4: a conclusion only holds if it survives the worst-case imputation as
  well as the complete-case analysis.
* **`validation_performance_loss` is diversity-discounted.** Discussed at the
  top. It is comparable across ambiguity settings but is not a measure of
  predictive quality on its own, which is why the task metric leads. Logging the
  raw weighted performance term separately in `math_utils.py` would remove the
  ambiguity here.
* **Early stopping is oracle early stopping.** The epoch is chosen on the same
  validation set that is then reported, so absolute numbers are optimistic in
  the usual way. Paired differences are far less affected, since the identical
  rule is applied to both members of a block.
* **Figures are overwritten on every run.** Re-executing replaces
  `paper_plots/*.pdf`, so regenerate before a submission rather than trusting
  whatever happens to be on disk.